# Subplot grids

**Many plots in one figure -- subplots, shared axes, and uneven grids.**

**What it shows:**

- plt.subplots(rows, cols) and the array of Axes it hands back
- sharex / sharey -- essential when panels must be compared
- subplot_mosaic for layouts that are not a plain grid
- tight_layout / constrained_layout, and when each one gives up

---

*Chapter:* `foundations` — matplotlib's actual mechanics  
*Run the cells in order.* Every figure is also written to `viz/output/foundations/`, which is what the Streamlit gallery (`viz/project/gallery.py`) reads.


## Setup

These lines are how every notebook in the folder finds `vizkit.py`, which holds the save helpers and the seeded sample data. The data is seeded on purpose: your figures should come out identical to everyone else's.

`save()` writes each figure into `viz/output/` **and** leaves it on screen here. The trailing `;` on those calls only stops the notebook echoing the path it returns.


In [ ]:
%matplotlib inline

# A notebook has no __file__, so find viz/ by walking up from this
# notebook's own folder until vizkit.py turns up.
import sys
from pathlib import Path

VIZ = next(p for p in [Path.cwd(), *Path.cwd().parents]
           if (p / "vizkit.py").exists())
sys.path.insert(0, str(VIZ))

import matplotlib.pyplot as plt
import numpy as np

from vizkit import save, sales

# Where save() files this lesson's output: viz/output/foundations/
LESSON = "foundations/subplots_grid"


## The data

Four regions of monthly sales, pivoted wide — one column per panel.


In [ ]:
data = sales()
wide = data.pivot(index="month", columns="region", values="sales")


## 1. Sharing axes is not cosmetic

Compare West across the two figures. Unshared, its panel is scaled to its own range, so a small series looks like a big one. Sharing the y axis is not tidiness; it is what makes the panels comparable at all.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, region in zip(axes, wide.columns):
    ax.plot(wide.index, wide[region], color="#4C72B0")
    ax.set_title(region)
    ax.tick_params(axis="x", rotation=45, labelsize=7)
fig.suptitle("NOT shared: each panel has its own y range. West looks like North.",
             fontsize=11)
fig.tight_layout()
save(fig, LESSON, "not-shared")

fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), sharey=True)
for ax, region in zip(axes, wide.columns):
    ax.plot(wide.index, wide[region], color="#4C72B0")
    ax.set_title(region)
    ax.tick_params(axis="x", rotation=45, labelsize=7)
axes[0].set_ylabel("sales")
fig.suptitle("sharey=True: now the panels are actually comparable", fontsize=11)
fig.tight_layout()
save(fig, LESSON, "shared");


## 2. Uneven layouts with subplot_mosaic

`subplot_mosaic` lets you draw the layout as text and get it back as a dict of Axes keyed by name. Repeated letters span cells, and named panels are far easier to read than `axes[0, 1]`.


In [ ]:
# Each string is a row; repeated letters make a panel span cells.
fig, axes = plt.subplot_mosaic(
    [["main", "main", "side"],
     ["main", "main", "side"],
     ["bottom", "bottom", "bottom"]],
    figsize=(10, 6),
)

for region in wide.columns:
    axes["main"].plot(wide.index, wide[region], label=region)
axes["main"].set_title("main: the headline chart")
axes["main"].legend(fontsize=8)

axes["side"].barh(wide.columns, wide.iloc[-1], color="#4C72B0")
axes["side"].set_title("side: latest month")

axes["bottom"].bar(wide.index, wide.sum(axis=1), width=20, color="#8172B2")
axes["bottom"].set_title("bottom: total across regions")

fig.suptitle("subplot_mosaic: draw the layout as text, get it as Axes",
             fontsize=12)
fig.tight_layout()
save(fig, LESSON, "mosaic");


## 3. Things that hang outside the axes

Four ways to deal with a legend hanging off the edge, acting at three different moments. Which ones you *need* depends on your matplotlib version — the comment records what was measured on 3.9, so re-measure rather than trusting folklore.


In [ ]:
# A legend anchored past the right edge is the classic case. There are three
# ways to deal with it, acting at three different moments -- and which ones
# you NEED depends on your matplotlib version, so measure rather than trust
# folklore. Measured here on matplotlib 3.9: with no layout call the legend is
# clipped (it ends 60px past the figure edge); tight_layout() already makes
# room for it. Older versions did not, which is where the "tight_layout can't
# handle legends" advice you will read online comes from.
rng = np.random.default_rng(0)
x = np.arange(30)


def six_series(ax):
    for i in range(6):
        ax.plot(x, np.cumsum(rng.normal(0, 1, 30)), label=f"series {i}")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")


# (a) no layout management at all -- the legend runs off the page.
#     crop=False, or the save would quietly rescue it and hide the problem.
fig, ax = plt.subplots(figsize=(6, 3))
six_series(ax)
ax.set_title("No layout call: the legend runs off the page")
save(fig, LESSON, "no-layout", crop=False)

# (b) fix at DRAW time, option 1: tight_layout, run once, now
fig, ax = plt.subplots(figsize=(6, 3))
six_series(ax)
ax.set_title("tight_layout(): room made for the legend")
fig.tight_layout()
save(fig, LESSON, "tight-layout", crop=False)

# (c) fix at DRAW time, option 2: constrained_layout, re-run whenever the
#     figure changes. Slower, but it copes with things added later.
fig, ax = plt.subplots(figsize=(6, 3), layout="constrained")
six_series(ax)
ax.set_title("constrained_layout: same, and it keeps up with changes")
save(fig, LESSON, "constrained", crop=False)

# (d) fix at SAVE time: grow the exported image to fit whatever is there.
#     This works even when the on-screen figure is wrong -- which is why
#     vizkit.save() does it for you by default.
fig, ax = plt.subplots(figsize=(6, 3))
six_series(ax)
ax.set_title('savefig(bbox_inches="tight"): export grown to fit')
save(fig, LESSON, "bbox-tight", crop=True);


## Rules of thumb

```text
Panels:
  comparing panels?          -> sharey=True, or the comparison is a lie
  layout is not a grid?      -> subplot_mosaic, drawn as text
  anything outside the axes? -> use a layout manager, or bbox_inches="tight"
                                when saving. Check your version -- do not
                                trust folklore about which one is broken.
```


## Try it yourself

Edit the cells above and re-run them — that is what the notebook is for.

1. In section 1, use `sharey=True` but give one region 10× its values. Does sharing still help, or does one panel now dominate?
2. Rewrite section 2's mosaic so `side` spans all three rows on the right. How many characters did that take?
3. Run section 3's four variants and open the saved PNGs. Which of them differ on *your* matplotlib version? Print `matplotlib.__version__` alongside your answer.


In [ ]:
# your turn


---

**Previous:** [`foundations/scales_and_ticks`](scales_and_ticks.ipynb)  
**Next:** [`foundations/saving`](saving.ipynb)
